# IOAI — 2026 Round2 Mix Functions (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-round2-mix-functions'
for f in ['train.csv','train_params.csv','test.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Mixed Functions — 모범답안

각 점이 어느 함수에서 나왔는지 모르므로 **EM 방식**으로 푼다: ① 4함수 파라미터를 초기화 → ② 각 점을 잔차가
가장 작은 함수에 배정 → ③ 함수별로 배정된 점에 `curve_fit` 재적합 → ②~③ 반복. 삼각함수 주파수 `c1` 은
여러 초기값으로 multi-start 해 국소최적을 피한다. 재구성 RMSE ≈ 0.12~0.3(노이즈 바닥 0.12).


In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import curve_fit

test = pd.read_csv("data/test.csv")
PCOLS = ["a_0","a_1","a_2","b_0","b_1","c_0","c_1","c_2","c_3","d_0","d_1"]

f_exp = lambda x,a0,a1,a2: a0*np.exp(a1*x)+a2
f_log = lambda x,b0,b1:    b0*np.log(np.abs(x)+1e-9)+b1
f_tri = lambda x,c0,c1,c2,c3: c0*np.sin(c1*x+c2)+c3
f_lin = lambda x,d0,d1:    d0*x+d1
FAMS = [(f_exp,slice(0,3),3),(f_log,slice(3,5),2),(f_tri,slice(5,9),4),(f_lin,slice(9,11),2)]

def resid(x,y,P):
    with np.errstate(all="ignore"):
        F=[f_exp(x,*P[0:3]),f_log(x,*P[3:5]),f_tri(x,*P[5:9]),f_lin(x,*P[9:11])]
    return np.nan_to_num(np.abs(np.vstack(F)-y),nan=1e9,posinf=1e9,neginf=1e9)

def solve(x,y,iters=12):
    x=np.asarray(x,float); y=np.asarray(y,float)
    d=np.polyfit(x,y,1); med=float(np.median(y)); best=None
    for c1 in [0.3,0.5,0.8,1.2,2.0]:                  # 삼각 주파수 multi-start
        P=[0.5,0.3,med, 1.0,med, 1.0,c1,0.0,med, float(d[0]),float(d[1])]
        for _ in range(iters):
            lab=resid(x,y,P).argmin(0); newP=list(P)
            for k,(f,sl,ng) in enumerate(FAMS):
                m=lab==k
                if m.sum()<ng+1: continue
                try: newP[sl],_=curve_fit(f,x[m],y[m],p0=P[sl],maxfev=4000)
                except Exception: pass
            P=newP
        r=np.sqrt((resid(x,y,P).min(0)**2).mean())
        if best is None or r<best[0]: best=(r,list(map(float,P)))
    return best[1]

rows=[]
for g,gg in test.groupby("group_id"):
    P=solve(gg["x"].values, gg["y"].values)
    rows.append({"group_id": g, **dict(zip(PCOLS,P))})
pd.DataFrame(rows)[["group_id"]+PCOLS].to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows), "groups")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)